# Sentiment Classification with LoRA Fine-Tuning (DistilBERT)

End-to-end project: exploratory data analysis, a classical ML baseline, parameter-efficient
LoRA fine-tuning of DistilBERT on the **full 50,000-review IMDB dataset**, a LoRA-rank
ablation study, full evaluation (confusion matrix, precision/recall/F1), training curves,
error analysis, model explainability (LIME), out-of-distribution robustness testing, and a
live interactive Gradio demo.

**Pipeline:**
1. Setup & environment fix
2. Load data + EDA (class balance, review length, word cloud)
3. Baseline model (TF-IDF + Logistic Regression)
4. LoRA fine-tuning of DistilBERT (full dataset)
5. Evaluation (metrics, confusion matrix, classification report)
6. Training curves
7. Error analysis (misclassified examples)
8. LoRA rank ablation study (parameter-efficiency trade-off)
9. Explainability with LIME (which words drive each prediction)
10. Out-of-distribution robustness test (sarcasm, negation, different domains)
11. Interactive inference demo (Gradio, shareable link)


## 1. Setup

Mount Drive and install dependencies. **Run this cell once**, then restart the runtime (`Runtime > Restart session`) before running the rest of the notebook — this avoids a known `torchao`/`peft` version conflict on Colab.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Install correct versions
!pip install -q transformers peft==0.13.2 accelerate pandas scikit-learn matplotlib seaborn wordcloud lime gradio
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip uninstall -y torchao -q

print("Setup complete. Now go to Runtime > Restart Session, then run the cells below (skip this one).")

## 2. Imports & Configuration

Run this after restarting the runtime.

In [ ]:
!pip uninstall -y torchao -q  # in case it reinstalls as a dependency after restart

import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_PATH = "/content/drive/MyDrive/IMDBDataset.csv"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
OUTPUT_DIR = "./my_lora_model"
SAVE_DIR = "./my_first_finetuned_model"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 3. Load Data & Exploratory Data Analysis

Using the **entire 50,000-review dataset** (no sampling) for training. Before training,
let's understand the data: class balance, review length distribution, and common words
in positive vs. negative reviews.

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["review", "sentiment"])
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})

print(f"Total reviews: {len(df)}")
print(df["sentiment"].value_counts())

# Class balance plot
plt.figure(figsize=(5, 4))
sns.countplot(x="sentiment", data=df, palette="Set2")
plt.title("Class Balance")
plt.savefig("class_balance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Review length distribution
df["review_length"] = df["review"].apply(lambda x: len(x.split()))

plt.figure(figsize=(7, 4))
sns.histplot(df["review_length"], bins=50, kde=True, color="steelblue")
plt.axvline(df["review_length"].median(), color="red", linestyle="--", label=f"Median: {df['review_length'].median():.0f} words")
plt.title("Review Length Distribution (word count)")
plt.xlabel("Word count")
plt.legend()
plt.savefig("review_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(df["review_length"].describe())

In [ ]:
# Word clouds for positive vs negative reviews
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, sentiment, color in zip(axes, ["positive", "negative"], ["Greens", "Reds"]):
    text = " ".join(df[df["sentiment"] == sentiment]["review"].sample(2000, random_state=SEED))
    wc = WordCloud(width=800, height=500, background_color="white",
                   colormap=color, stopwords={"br", "movie", "film"}).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"{sentiment.capitalize()} Reviews - Common Words", fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.savefig("wordclouds.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Train / Validation / Test Split

Full dataset split with stratification to preserve class balance across all three sets.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")

## 5. Baseline Model — TF-IDF + Logistic Regression

Before fine-tuning a transformer, it's good practice to establish a simple, fast baseline.
This tells us how much lift the LoRA-tuned DistilBERT actually provides, rather than just
reporting a number in isolation.

In [ ]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(train_df["review"])
X_test_tfidf = tfidf.transform(test_df["review"])

baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train_tfidf, train_df["label"])

baseline_preds = baseline.predict(X_test_tfidf)
baseline_acc = accuracy_score(test_df["label"], baseline_preds)
baseline_f1 = f1_score(test_df["label"], baseline_preds)

print(f"Baseline (TF-IDF + Logistic Regression)")
print(f"  Accuracy: {baseline_acc:.4f}")
print(f"  F1 Score: {baseline_f1:.4f}")

## 6. Tokenizer & PyTorch Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = SentimentDataset(train_df["review"].tolist(), train_df["label"].tolist(), tokenizer)
val_dataset = SentimentDataset(val_df["review"].tolist(), val_df["label"].tolist(), tokenizer)
test_dataset = SentimentDataset(test_df["review"].tolist(), test_df["label"].tolist(), tokenizer)

## 7. Model + LoRA Configuration

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Training Configuration

Trains on the **full dataset** with early stopping (patience=2) so training halts once
validation F1 stops improving, avoiding wasted epochs and overfitting.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

## 9. Train

On the full 40k training set this will take noticeably longer than the earlier 960-sample version — that's expected and is part of what makes this a real training run rather than a quick demo.

In [ ]:
print("Training started...")
train_result = trainer.train()
print("Training complete.")
print(train_result.metrics)

## 10. Final Evaluation on Held-Out Test Set

The test set was never seen during training or model selection (validation was used for early stopping / best-checkpoint selection), so this is an unbiased estimate of real-world performance.

In [ ]:
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=-1)
test_labels = test_df["label"].values

test_acc = accuracy_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="binary")

print("=== LoRA Fine-Tuned DistilBERT — Test Set Results ===")
print(f"Accuracy:  {test_acc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print()
print(f"=== Baseline (TF-IDF + LogisticRegression) for comparison ===")
print(f"Accuracy:  {baseline_acc:.4f}")
print(f"F1 Score:  {baseline_f1:.4f}")
print()
print(f"Improvement over baseline: {(test_acc - baseline_acc) * 100:+.2f} accuracy points")

print("\nFull classification report:")
print(classification_report(test_labels, test_preds, target_names=["negative", "positive"]))

# Save metrics to a JSON file for the README/portfolio
metrics_summary = {
    "lora_finetuned": {"accuracy": test_acc, "f1": f1, "precision": precision, "recall": recall},
    "baseline_tfidf_logreg": {"accuracy": baseline_acc, "f1": baseline_f1},
}
with open("test_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("\nSaved metrics_summary to test_metrics.json")

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "positive"], yticklabels=["negative", "positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Test Set")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Training Curves

Loss and F1 score across epochs, extracted from the Trainer's log history.

In [ ]:
log_history = trainer.state.log_history

train_loss = [(x["epoch"], x["loss"]) for x in log_history if "loss" in x]
eval_f1 = [(x["epoch"], x["eval_f1"]) for x in log_history if "eval_f1" in x]
eval_loss = [(x["epoch"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_loss:
    epochs_t, losses_t = zip(*train_loss)
    axes[0].plot(epochs_t, losses_t, label="Train Loss", color="steelblue")
if eval_loss:
    epochs_e, losses_e = zip(*eval_loss)
    axes[0].plot(epochs_e, losses_e, label="Validation Loss", color="darkorange", marker="o")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training vs Validation Loss")
axes[0].legend()

if eval_f1:
    epochs_f, f1s = zip(*eval_f1)
    axes[1].plot(epochs_f, f1s, label="Validation F1", color="seagreen", marker="o")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 Score")
axes[1].set_title("Validation F1 Score over Training")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Error Analysis

Looking at *what* the model gets wrong is often more informative than the aggregate
metrics above — it reveals whether errors are random noise or a systematic pattern
(e.g. sarcasm, mixed-sentiment reviews, very short reviews).

In [ ]:
test_df_analysis = test_df.copy().reset_index(drop=True)
test_df_analysis["predicted"] = test_preds
test_df_analysis["correct"] = test_df_analysis["label"] == test_df_analysis["predicted"]

misclassified = test_df_analysis[~test_df_analysis["correct"]]
print(f"Misclassified: {len(misclassified)} / {len(test_df_analysis)} ({len(misclassified)/len(test_df_analysis)*100:.2f}%)")

print("\n--- Sample misclassified reviews ---\n")
label_map = {0: "negative", 1: "positive"}
for _, row in misclassified.sample(min(5, len(misclassified)), random_state=SEED).iterrows():
    print(f"True: {label_map[row['label']]} | Predicted: {label_map[row['predicted']]}")
    print(f"Review (truncated): {row['review'][:250]}...")
    print("-" * 80)

## 14. LoRA Rank Ablation Study

A core question with LoRA is: *how small can the adapter be before performance suffers?*
Here we retrain with a few different LoRA ranks (`r=4, 8, 16`) on a fixed subset (for speed)
and compare trainable-parameter count against validation F1. This is the kind of trade-off
analysis that shows *why* a hyperparameter was chosen, not just that it was.

> Uses a 6,000-review subset with 2 epochs per configuration purely to keep ablation runtime
> reasonable — the main model above was trained on the full dataset.

In [ ]:
ablation_subset = df.sample(6000, random_state=SEED)
ab_train_df, ab_val_df = train_test_split(
    ablation_subset, test_size=0.2, random_state=SEED, stratify=ablation_subset["label"]
)
ab_train_df = ab_train_df.reset_index(drop=True)
ab_val_df = ab_val_df.reset_index(drop=True)

ab_train_dataset = SentimentDataset(ab_train_df["review"].tolist(), ab_train_df["label"].tolist(), tokenizer)
ab_val_dataset = SentimentDataset(ab_val_df["review"].tolist(), ab_val_df["label"].tolist(), tokenizer)

ablation_results = []

for rank in [4, 8, 16]:
    print(f"\n=== Training with LoRA r={rank} ===")
    base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    ab_lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=rank, lora_alpha=rank * 2,
        lora_dropout=0.1, target_modules=["q_lin", "v_lin"]
    )
    ab_model = get_peft_model(base_model, ab_lora_config)

    trainable_params = sum(p.numel() for p in ab_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in ab_model.parameters())

    ab_args = TrainingArguments(
        output_dir=f"./ablation_r{rank}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        logging_steps=100,
        report_to="none",
        disable_tqdm=True,
    )

    ab_trainer = Trainer(
        model=ab_model, args=ab_args,
        train_dataset=ab_train_dataset, eval_dataset=ab_val_dataset,
        compute_metrics=compute_metrics,
    )
    ab_trainer.train()
    ab_metrics = ab_trainer.evaluate()

    ablation_results.append({
        "rank": rank,
        "trainable_params": trainable_params,
        "trainable_pct": trainable_params / total_params * 100,
        "val_f1": ab_metrics["eval_f1"],
        "val_accuracy": ab_metrics["eval_accuracy"],
    })
    print(f"r={rank}: {trainable_params:,} trainable params ({trainable_params/total_params*100:.3f}%), val F1={ab_metrics['eval_f1']:.4f}")

ablation_df = pd.DataFrame(ablation_results)
print("\n", ablation_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(ablation_df["rank"].astype(str), ablation_df["trainable_params"], color="mediumpurple")
axes[0].set_xlabel("LoRA rank (r)")
axes[0].set_ylabel("Trainable parameters")
axes[0].set_title("Trainable Parameters vs. LoRA Rank")

axes[1].plot(ablation_df["rank"], ablation_df["val_f1"], marker="o", color="darkorange")
axes[1].set_xlabel("LoRA rank (r)")
axes[1].set_ylabel("Validation F1")
axes[1].set_title("Validation F1 vs. LoRA Rank")
axes[1].set_xticks(ablation_df["rank"])

plt.tight_layout()
plt.savefig("lora_rank_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

ablation_df.to_json("ablation_results.json", orient="records", indent=2)

## 15. Explainability — Which Words Drive Each Prediction?

Aggregate metrics tell us *how often* the model is right; LIME tells us *why* it made a
specific prediction, by perturbing the input text and observing how the predicted
probability changes. This is useful both for debugging (catching a model that's
right for the wrong reasons — e.g. keying off a director's name instead of sentiment
words) and for presenting the model's behavior to a non-technical audience.

In [ ]:
from lime.lime_text import LimeTextExplainer

class_names = ["negative", "positive"]
explainer = LimeTextExplainer(class_names=class_names)

def lime_predict_proba(texts):
    """LIME needs a function that returns class probabilities for a list of texts."""
    model.eval()
    device = next(model.parameters()).device
    all_probs = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True,
                                padding=True, max_length=MAX_LENGTH)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
            all_probs.append(probs)
    return np.array(all_probs)

# Explain a couple of test-set predictions
sample_indices = test_df.sample(2, random_state=SEED).index
for idx in sample_indices:
    text = test_df.loc[idx, "review"][:500]  # LIME re-runs the model many times; keep it short
    true_label = label_map[test_df.loc[idx, "label"]]
    print(f"\n{'='*80}\nTrue label: {true_label}\nText: {text[:200]}...\n")
    exp = explainer.explain_instance(text, lime_predict_proba, num_features=8, num_samples=200)
    fig = exp.as_pyplot_figure()
    plt.tight_layout()
    plt.savefig(f"lime_explanation_{idx}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 16. Out-of-Distribution Robustness Test

The model was trained exclusively on IMDB movie reviews. Real-world deployment often means
encountering text that differs from the training distribution — different domains, sarcasm,
negation, or mixed sentiment. This section stress-tests the model on hand-written examples
outside its training distribution to get an honest sense of where it might struggle,
rather than only reporting in-distribution test accuracy.

In [ ]:
ood_examples = [
    # Negation
    ("This is not a bad movie at all, actually quite enjoyable.", "positive"),
    ("I wouldn't say it was good.", "negative"),
    # Sarcasm
    ("Oh great, another two hours of my life I'll never get back.", "negative"),
    ("Wow, truly a 'masterpiece' if your definition of masterpiece is boring.", "negative"),
    # Different domain: restaurant review
    ("The food was cold and the service was painfully slow.", "negative"),
    ("Best pasta I've had in years, the staff were lovely too.", "positive"),
    # Different domain: product review
    ("The battery died within a day and customer support never responded.", "negative"),
    ("Works exactly as described, arrived early, very happy with this purchase.", "positive"),
    # Mixed sentiment
    ("The acting was superb but the plot was a complete mess.", "negative"),
]

print(f"{'Text':<70} {'True':<10} {'Predicted':<10} {'Confidence'}")
print("-" * 105)
ood_correct = 0
model.eval()
device = next(model.parameters()).device
for text, true_label in ood_examples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        pred_id = int(torch.argmax(probs))
    pred_label = label_map[pred_id]
    ood_correct += int(pred_label == true_label)
    display_text = text if len(text) <= 67 else text[:67] + "..."
    print(f"{display_text:<70} {true_label:<10} {pred_label:<10} {float(probs[pred_id]):.2%}")

print(f"\nOOD accuracy: {ood_correct}/{len(ood_examples)} ({ood_correct/len(ood_examples)*100:.1f}%)")
print("Note: this is a small, hand-picked stress test, not a statistically rigorous benchmark —")
print("its purpose is qualitative insight into failure modes (negation, sarcasm, domain shift), not a headline metric.")

## 17. Save the Fine-Tuned Model

In [ ]:
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

# Also copy plots + metrics to Drive so they survive the Colab session ending
import shutil
os.makedirs("/content/drive/MyDrive/sentiment_lora_artifacts", exist_ok=True)
for f in ["class_balance.png", "review_length_distribution.png", "wordclouds.png",
          "confusion_matrix.png", "training_curves.png", "test_metrics.json",
          "lora_rank_ablation.png", "ablation_results.json"]:
    if os.path.exists(f):
        shutil.copy(f, f"/content/drive/MyDrive/sentiment_lora_artifacts/{f}")
print("Artifacts backed up to Google Drive: sentiment_lora_artifacts/")

## 18. Inference Demo

Quick sanity check on hand-written sentences, including a couple of harder/ambiguous cases.

In [ ]:
def predict_sentiment(text, model, tokenizer, max_length=MAX_LENGTH):
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        pred_id = int(torch.argmax(probs))
    label = "positive" if pred_id == 1 else "negative"
    return label, float(probs[pred_id])

demo_sentences = [
    "This movie was an absolute masterpiece, I was captivated from start to finish!",
    "Waste of two hours. The plot made no sense and the acting was wooden.",
    "It wasn't terrible, but I wouldn't watch it again either.",
    "I expected to hate it, but somehow it grew on me by the end.",
]

for sentence in demo_sentences:
    label, confidence = predict_sentiment(sentence, model, tokenizer)
    print(f"Text: {sentence}")
    print(f"  -> Prediction: {label} (confidence: {confidence:.2%})\n")

## 19. Interactive Live Demo (Gradio)

Running this cell launches a small web app with a shareable public link (valid for 72
hours) — useful for dropping into a portfolio, resume, or LinkedIn post so people can try
the model themselves instead of just reading about it.

In [ ]:
import gradio as gr

def gradio_predict(text):
    label, confidence = predict_sentiment(text, model, tokenizer)
    return {"positive": confidence if label == "positive" else 1 - confidence,
            "negative": confidence if label == "negative" else 1 - confidence}

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=4, placeholder="Type a movie review here...", label="Review Text"),
    outputs=gr.Label(num_top_classes=2, label="Predicted Sentiment"),
    title="IMDB Sentiment Classifier (LoRA-tuned DistilBERT)",
    description="A DistilBERT model fine-tuned with LoRA on 50,000 IMDB reviews. Try a review below.",
    examples=[
        ["This film changed the way I see cinema. Absolutely stunning."],
        ["I fell asleep twice. Painfully slow and pointless."],
        ["Not the best, not the worst — a solidly average watch."],
    ],
)

demo.launch(share=True, debug=False)

## Summary

- Trained on the **full 50,000-review IMDB dataset** (not a small sample)
- LoRA fine-tuning updates a small fraction of DistilBERT's parameters, shown via `print_trainable_parameters()`
- Outperforms a TF-IDF + Logistic Regression baseline (see Section 5 vs. Section 10)
- Early stopping based on validation F1 prevents overfitting and wasted compute
- Full evaluation: confusion matrix, precision/recall/F1, classification report
- Error analysis identifies where the model struggles, not just aggregate accuracy
- LoRA rank ablation study quantifies the parameter-efficiency vs. performance trade-off
- LIME explainability shows *why* the model made specific predictions, not just accuracy
- Out-of-distribution testing gives an honest read on generalization limits (sarcasm, negation, domain shift)
- A live Gradio demo lets anyone interact with the model directly
- All plots and metrics are saved to Google Drive for reuse in a README / portfolio write-up
